# 🧹 Day 7 — Data Cleaning Pipeline (8 Stages)

This notebook walks through the same 8 stages as `clean_articles.py` — step by step so you can **see exactly what each stage does**.

**The 3 Principles we follow:**
- **P1** — Never overwrite raw data. Every change creates a NEW column.
- **P2** — Never silently delete rows. Bad rows get a reason and go to a rejected file.
- **P3** — No data leakage. Nothing is fitted on the data.

---
## 📦 Setup — Import Libraries

In [1]:
import re, html, hashlib, unicodedata, os
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit, parse_qsl, urlencode
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

print('✅ All libraries imported')
print(f'   pandas  {pd.__version__}')
print(f'   numpy   {np.__version__}')

✅ All libraries imported
   pandas  3.1.0
   numpy   2.4.6


---
## ⚙️ Configuration

All thresholds and settings live here. **Change a number here, everything below follows.**

In [ ]:
INPUT_FILE = r"C:\Users\talha\OneDrive\Desktop\Limox\day 4\output_bulk\final\articles_bulk.csv"
SAMPLE = 500   # set to None to run on all ~39k rows

COLUMN_ALIASES = {
    'url':            ['url', 'link', 'article_url'],
    'title':          ['title', 'headline', 'article_title'],
    'body':           ['body', 'text', 'content', 'article_text', 'full_text'],
    'published_date': ['published_date', 'date', 'published', 'pub_date'],
    'source':         ['source', 'site', 'domain', 'publisher'],
    'author':         ['author', 'authors', 'byline'],
}

NULL_LIKE          = ['', 'n/a', 'na', 'null', 'none', 'nan', 'nil', '-', '--', 'unknown']
MIN_BODY_CHARS     = 200
MIN_TITLE_CHARS    = 5
NEAR_DUP_THRESHOLD = 0.80
QUALITY_GOOD       = 0.85
QUALITY_REVIEW     = 0.60
TRACKING_PREFIXES  = ['utm_', 'fbclid', 'gclid', 'ref', 'mc_', 'igshid']
BOILERPLATE_PHRASES = [
    'subscribe now', 'sign up for our newsletter', 'follow us on',
    'all rights reserved', 'cookie policy', 'accept cookies',
    'advertisement', 'click here to', 'share this article',
    'related articles', 'privacy policy', 'terms of use',
]

print('✅ Configuration loaded')

---
## 🔵 Stage 1 — Ingestion
> **Goal:** Load the CSV and keep a raw backup. Never touch the original again.

We also add two helper columns:
- `row_id` — a stable unique number for every row
- `ingested_at` — timestamp when we loaded the data

In [ ]:
df_raw = pd.read_csv(INPUT_FILE)
print(f'📥 Loaded {len(df_raw):,} rows from {Path(INPUT_FILE).name}')

if SAMPLE:
    df = df_raw.sample(n=min(SAMPLE, len(df_raw)), random_state=42).copy()
    df = df.reset_index(drop=True)
    print(f'🔬 Using sample of {len(df):,} rows')
else:
    df = df_raw.copy()

# P1: Add stable row IDs and ingestion timestamp (new columns only, never overwrite)
df.insert(0, 'row_id', range(len(df)))
df['ingested_at'] = datetime.now(timezone.utc).isoformat()

print(f'\n📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns')
df.head(3)

---
## 🔵 Stage 2 — Profiling
> **Goal:** Inspect the data — types, nulls, unique values. Just **observation**, nothing changes.

Think of this as a doctor's checkup before doing any surgery.

In [ ]:
print(f'Rows: {len(df):,}   Columns: {df.shape[1]}   Memory: {df.memory_usage(deep=True).sum()/1e6:.1f} MB\n')

profile = pd.DataFrame({
    'dtype':       df.dtypes,
    'null_count':  df.isnull().sum(),
    'null_%':      (df.isnull().sum() / len(df) * 100).round(1),
    'unique_vals': df.nunique(),
    'example':     df.iloc[0]
})
profile

In [ ]:
# Text length stats — how long are titles? how long are article bodies?
str_cols = [c for c in df.columns if df[c].dtype == object]
df[str_cols].apply(lambda col: col.fillna('').str.len()).describe(
    percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]
).T[['count','mean','min','50%','95%','max']].round(1)

---
## 🔵 Stage 3 — Standardization
> **Goal:** Make everything consistent — same column names, same date format, clean URLs.

1. Map your column names to standard names (`text` → `body`)
2. Strip tracking params from URLs (`?utm_source=twitter` → removed)
3. Parse and normalize dates to ISO format (UTC)
4. Add `domain` and `source_std` columns

In [ ]:
def find_col(df, candidates):
    cols_lower = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in cols_lower:
            return cols_lower[c.lower()]
    return None

col_map = {std: find_col(df, aliases) for std, aliases in COLUMN_ALIASES.items()}
print('Column mapping:')
for std, actual in col_map.items():
    status = '✅' if actual else '❌ NOT FOUND'
    print(f'  {std:18s} → {actual or "MISSING"} {status}')

In [ ]:
# P1: Keep raw originals before any changes
for std, actual in col_map.items():
    if actual and actual in df.columns:
        df[f'raw_{std}'] = df[actual].copy()  # creates raw_url, raw_title, raw_body etc.
        df[std] = df[actual]                  # creates standard alias column

# Clean URL: remove tracking params
def clean_url(u):
    if not isinstance(u, str): return u
    try:
        p = urlsplit(u.strip())
        q = [(k,v) for k,v in parse_qsl(p.query) if not any(k.lower().startswith(x) for x in TRACKING_PREFIXES)]
        return urlunsplit(p._replace(query=urlencode(q)))
    except: return u

df['url_norm']   = df['url'].apply(clean_url)
df['domain']     = df['url_norm'].apply(lambda u: urlsplit(str(u)).netloc.replace('www.','') if isinstance(u,str) else '')
df['source_std'] = df['source'].fillna('').str.lower().str.replace(r'[^a-z0-9]+','_',regex=True).str.strip('_')
df['date_iso']   = pd.to_datetime(df['published_date'], errors='coerce', utc=True)

print(f'Date range: {df["date_iso"].min().date()} → {df["date_iso"].max().date()}')
print(f'Parsed dates: {df["date_iso"].notna().sum():,} / {len(df):,}')

---
## 🔵 Stage 4 — Validation
> **Goal:** Check every row against rules. Flag problems. Only **reject** rows that are completely unusable.

**P2 in action:** We do NOT silently delete rows. Each bad row gets a `reject_reason` and is saved so you can inspect it later.

In [ ]:
body_col = col_map.get('body')

def is_null_like(val):
    if val is None or (isinstance(val, float) and np.isnan(val)): return True
    return str(val).strip().lower() in NULL_LIKE

flags = pd.DataFrame(index=df.index)
flags['missing_url']   = df['url'].apply(is_null_like)
flags['invalid_url']   = ~df['url'].fillna('').str.match(r'^https?://', re.I)
flags['missing_date']  = df['date_iso'].isna()
flags['missing_title'] = df['title'].apply(is_null_like)
flags['short_title']   = df['title'].fillna('').str.len() < MIN_TITLE_CHARS
flags['empty_body']    = df[body_col].apply(is_null_like) if body_col else True
flags['short_body']    = (~flags['empty_body']) & (df[body_col].fillna('').str.len() < MIN_BODY_CHARS) if body_col else False
flags['invalid_url_and_no_body'] = flags['invalid_url'] & flags['empty_body']

print('Validation Rule Results:')
print('=' * 45)
for rule, series in flags.items():
    count = int(series.sum())
    pct = count / len(df) * 100
    icon = '🔴' if count > 0 else '✅'
    print(f'  {icon} {rule:<35s} {count:>5,} rows  ({pct:.1f}%)')

In [ ]:
REJECT_RULES = ['empty_body', 'invalid_url_and_no_body']

df['is_rejected']   = flags[REJECT_RULES].any(axis=1)
df['reject_reason'] = ''
for rule in REJECT_RULES:
    mask = flags[rule] & (df['reject_reason'] == '')
    df.loc[mask, 'reject_reason'] = rule

non_reject_flags = [c for c in flags.columns if c not in REJECT_RULES]
df['flags'] = flags[non_reject_flags].apply(
    lambda row: '|'.join(col for col in non_reject_flags if row[col]), axis=1
)

print(f'✅ Kept:     {(~df["is_rejected"]).sum():,} rows')
print(f'❌ Rejected: {df["is_rejected"].sum():,} rows')

---
## 🔵 Stage 5 — Deduplication
> **Goal:** Find articles that are the same or very similar and keep only one copy.

1. **Exact duplicates** — same URL or identical text hash
2. **Near duplicates** — 80%+ similar text (Jaccard similarity on word shingles)

In [ ]:
# Exact URL duplicates
url_dupes = df.duplicated(subset=['url_norm'], keep='first')
print(f'Exact URL duplicates: {url_dupes.sum():,}')

# Exact content hash duplicates
if body_col:
    df['content_hash'] = df[body_col].fillna('').apply(
        lambda t: hashlib.md5(t.encode('utf-8', errors='ignore')).hexdigest()
    )
    hash_dupes = df.duplicated(subset=['content_hash'], keep='first')
    print(f'Exact content duplicates: {hash_dupes.sum():,}')
else:
    hash_dupes = pd.Series(False, index=df.index)

exact_dupe_mask = url_dupes | hash_dupes
print(f'Total exact duplicates: {exact_dupe_mask.sum():,}')

In [ ]:
# Near-duplicates using Jaccard similarity on word shingles
# (Simplified version — full script uses MinHash+LSH for speed at scale)
def shingles(text, n=5):
    words = str(text).lower().split()
    return set(' '.join(words[i:i+n]) for i in range(max(0, len(words)-n+1)))

def jaccard(a, b):
    if not a or not b: return 0.0
    return len(a & b) / len(a | b)

near_dup_pairs = []
if body_col:
    # Demo on small subset to keep it fast in notebook
    demo_df = df[~df['is_rejected']].head(200)
    shingled = demo_df[body_col].apply(lambda t: shingles(t, 5))
    ids = demo_df.index.tolist()
    for i in range(len(ids)):
        for j in range(i+1, len(ids)):
            sim = jaccard(shingled.iloc[i], shingled.iloc[j])
            if sim >= NEAR_DUP_THRESHOLD:
                near_dup_pairs.append((ids[i], ids[j], round(sim,3)))

    print(f'Near-duplicate pairs found (≥{NEAR_DUP_THRESHOLD}): {len(near_dup_pairs)}')
    if near_dup_pairs:
        i, j, sim = near_dup_pairs[0]
        print(f'\nExample pair (similarity={sim}):')
        print(f'  A: {df.loc[i,"title"][:80]}')
        print(f'  B: {df.loc[j,"title"][:80]}')

near_dup_idx = set(j for _, j, _ in near_dup_pairs)
df['is_duplicate'] = exact_dupe_mask | df.index.isin(near_dup_idx)
df['duplicate_group_id'] = pd.NA
df['duplicate_group_size'] = pd.NA
print(f'\nTotal marked duplicate: {df["is_duplicate"].sum():,}')
print(f'Unique articles left:   {(~df["is_duplicate"] & ~df["is_rejected"]).sum():,}')

---
## 🔵 Stage 6 — Cleaning Text
> **Goal:** Clean the raw article text — remove HTML, decode special characters, strip boilerplate.

The **original text** is always kept in `raw_body`. The clean version goes into `body_clean` (P1).

| Step | What it does |
|---|---|
| Strip HTML | Removes `<p>`, `<b>`, `<div>` etc. |
| Decode entities | `&amp;` → `&`, `&nbsp;` → space |
| Normalize unicode | Fix weird characters |
| Remove boilerplate | Delete 'Subscribe now', 'Follow us on Twitter' etc. |
| Fix whitespace | Collapse multiple spaces/newlines |

In [ ]:
# Demonstrate HTML stripping
example_html = '<p>This is an <b>article</b> about <a href="#">Pakistan</a>.</p><br/>Follow us on Twitter!'
cleaned = BeautifulSoup(example_html, 'lxml').get_text(separator=' ')
print('BEFORE:', example_html)
print('AFTER: ', cleaned)

In [ ]:
# Demonstrate entity decoding
example_entities = 'Pakistan &amp; India signed agreement.&nbsp; Click here to read more!'
decoded = html.unescape(example_entities)
print('BEFORE:', example_entities)
print('AFTER: ', decoded)

In [ ]:
# Demonstrate boilerplate removal
example_bp = '''Karachi: The government announced new policies today.
Share this article
Subscribe now to get more news!
The minister said the reforms would help the economy.
Follow us on Twitter and Facebook.
All rights reserved 2026.'''

def remove_boilerplate(text):
    if not isinstance(text, str): return text
    lines = text.splitlines()
    return '\n'.join(l for l in lines if not any(p in l.lower() for p in BOILERPLATE_PHRASES))

print('BEFORE:'); print(example_bp)
print('\nAFTER:');  print(remove_boilerplate(example_bp))

In [ ]:
# Apply all cleaning to the full dataframe
def clean_text(text):
    if not isinstance(text, str): return ''
    text = BeautifulSoup(text, 'lxml').get_text(separator=' ')
    text = html.unescape(text)
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = remove_boilerplate(text)
    return text.strip()

if body_col:
    print('Cleaning text...')
    df['body_clean']    = df[body_col].apply(clean_text)
    df['title_clean']   = df['title'].fillna('').apply(lambda t: re.sub(r'\s+', ' ', str(t)).strip())
    df['chars_removed'] = (df[body_col].fillna('').str.len() - df['body_clean'].str.len()).clip(lower=0)
    df['body_chars']    = df['body_clean'].str.len()
    df['body_words']    = df['body_clean'].apply(lambda t: len(str(t).split()))
    print(f'✅ Done! Avg chars removed: {df["chars_removed"].mean():.0f}')
    print(f'\nBody length after cleaning:')
    print(df['body_chars'].describe().round(0).astype(int).to_string())
else:
    df['body_clean'] = ''
    df['title_clean'] = df['title'].fillna('')
    df['body_chars'] = 0
    df['body_words'] = 0
    print('⚠️  No body column found')

In [ ]:
# Show before vs after on a real article
if body_col:
    row = df[df['body_chars'] > 500].iloc[0]
    print('=== RAW TEXT (first 400 chars) ===')
    print(str(row[body_col])[:400])
    print('\n=== CLEAN TEXT (first 400 chars) ===')
    print(str(row['body_clean'])[:400])

---
## 🔵 Stage 7 — Quality Assessment
> **Goal:** Give every article a quality score from 0.0 to 1.0, and a grade: `good`, `review`, or `poor`.

| Grade | Score | Meaning |
|---|---|---|
| ✅ `good` | ≥ 0.85 | Ready to use |
| ⚠️ `review` | 0.60–0.84 | Usable but check manually |
| ❌ `poor` | < 0.60 | Low quality |

In [ ]:
# Completeness score: how many required fields does this row have?
checks = {
    'has_url':    df['url'].fillna('').str.len() > 0,
    'has_title':  df['title'].fillna('').str.len() >= MIN_TITLE_CHARS,
    'has_body':   df['body_clean'].fillna('').str.len() >= MIN_BODY_CHARS,
    'has_date':   df['date_iso'].notna(),
    'has_author': df['author'].fillna('').str.len() > 0,
}

df['completeness']   = pd.DataFrame(checks).mean(axis=1).round(4)
df['quality_score']  = df['completeness'].copy()
df['quality_flag']   = np.select(
    [df['quality_score'] >= QUALITY_GOOD, df['quality_score'] >= QUALITY_REVIEW],
    ['good', 'review'], default='poor'
)
df['quality_flags'] = ''

print('Completeness per field:')
for field, series in checks.items():
    pct = series.mean() * 100
    bar = '█' * int(pct / 5)
    print(f'  {field:<15s} {pct:5.1f}%  {bar}')

print(f'\nQuality grades:')
for grade, count in df['quality_flag'].value_counts().items():
    icon = {'good':'✅','review':'⚠️','poor':'❌'}.get(grade,'?')
    print(f'  {icon} {grade:<10s} {count:>5,}  ({count/len(df)*100:.1f}%)')

---
## 🔵 Stage 8 — Export
> **Goal:** Save results to separate files — clean articles, rejected rows, duplicates.

| File | Contents |
|---|---|
| `articles_clean.parquet` | ✅ Clean articles — use this for analysis |
| `articles_rejected.csv` | ❌ Rows that failed validation (with reason) |
| `articles_duplicates.csv` | 🔁 Duplicate rows that were removed |

In [ ]:
# Split into clean / rejected / duplicates
rejected   = df[df['is_rejected']].copy()
duplicates = df[(~df['is_rejected']) & df['is_duplicate']].copy()
clean      = df[(~df['is_rejected']) & (~df['is_duplicate'])].copy()

print('Final split:')
print(f'  ✅ Clean:      {len(clean):>6,}')
print(f'  ❌ Rejected:   {len(rejected):>6,}')
print(f'  🔁 Duplicates: {len(duplicates):>6,}')
print(f'  📊 Total:      {len(df):>6,}')

In [ ]:
OUT_DIR = Path(r'C:\Users\talha\OneDrive\Desktop\Limox\day 7\data\notebook_clean')
OUT_DIR.mkdir(parents=True, exist_ok=True)

clean.to_parquet(OUT_DIR / 'articles_clean.parquet', index=False)
print(f'✅ Saved {len(clean):,} clean articles to articles_clean.parquet')

if len(rejected) > 0:
    rejected.to_csv(OUT_DIR / 'articles_rejected.csv', index=False)
    print(f'❌ Saved {len(rejected):,} rejected rows')
else:
    print('✅ No rejected rows!')

if len(duplicates) > 0:
    duplicates[['row_id','url_norm','title','source']].to_csv(OUT_DIR / 'articles_duplicates.csv', index=False)
    print(f'🔁 Saved {len(duplicates):,} duplicate rows')

---
## 📊 Final Summary

In [ ]:
print('=== PIPELINE COMPLETE ===')
print(f'Input rows:    {len(df):>6,}')
print(f'Clean rows:    {len(clean):>6,}  ({len(clean)/len(df)*100:.1f}%)')
print(f'Rejected:      {len(rejected):>6,}')
print(f'Duplicates:    {len(duplicates):>6,}')
print(f'\nQuality breakdown:')
print(clean['quality_flag'].value_counts().to_string())
print(f'\nTop 10 sources:')
print(clean['source_std'].value_counts().head(10).to_string())

In [ ]:
clean[['title_clean','source_std','date_iso','body_words','quality_flag']].head(10)

In [ ]:
# Look at a single clean article in full
row = clean.iloc[0]
print(f'Title:   {row["title_clean"]}')
print(f'Source:  {row["source_std"]}')
print(f'Date:    {row["date_iso"]}')
print(f'Words:   {row["body_words"]}')
print(f'Quality: {row["quality_flag"]} (score={row["quality_score"]})')
print(f'\nClean body (first 600 chars):')
print(str(row.get('body_clean',''))[:600])